# Becke Partition 简单理解

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
grids = dft.grid.Grids(mol)
grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
grids.build(sort_grids=False)

## PySCF 格点生成过程

PySCF 的格点生成是
- 先生成原子格点与其权重；
- 随后依据 Becke partition 重新分配权重。

我们原则上只关心后一个过程，但 PySCF 的外部函数不会直接给出重新分配权重的缩放情况。因此，我们仍然需要先生成原始的原子格点与其权重。

同时，留意我们将实现狭义的 Becke partition。其 radii 的计算过程在 PySCF 中对应的函数是 `becke_atomic_radii_adjust`。PySCF 默认的 becke_scheme 就是 1988 年的版本，但后来有 Stratmann 与 Ochsenfeld 的改进版本。

In [4]:
atom_grids_tab = grids.gen_atomic_grids(mol)
coords, weights = grids.get_partition(mol, atom_grids_tab, radii_adjust=dft.radi.becke_atomic_radii_adjust)

## Becke Partition 实现与公式对应

### RADII Table

在实际计算前，我们需要先生成 RADII 矫正表。

REST 目前的实现是 dftlibs/numgrid 库。该库在这方面并不好，是有改进空间的。这个表格原理上是预先存储的，现算不是不行但没必要。

In [5]:
natm = mol.natm
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
fac_radii = [becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]
fac_radii = np.array(fac_radii).reshape(natm, natm)

In [6]:
fac_radii

array([[ 0.     , -0.32967, -0.32967, -0.32967],
       [ 0.32967,  0.     ,  0.     ,  0.     ],
       [ 0.32967,  0.     ,  0.     ,  0.     ],
       [ 0.32967,  0.     ,  0.     ,  0.     ]])

从具体的实现来说，首先，该表格可以一次性执行完成，因此我们可以不用关注效率。(非常小的 $O(N^2)$)。那么我们就不需要费心搞向量化计算流程了。在当前例子里，我们现在只需要考虑 N-H 原子的 RADII 矫正。

$$
\begin{align}
u_{AB} &= \frac{r_A - r_B}{r_A + r_B} \tag{A6} \\
a_{AB} &= \frac{u_{AB}}{u_{AB}^2 - 1} \tag{A5} \\
\Vert a_{AB} \Vert &\leqslant \frac{1}{2} \tag{A3}
\end{align}
$$

In [7]:
rA = grids.atomic_radii[7]
rB = grids.atomic_radii[1]
mu = (rA - rB) / (rA + rB)  # eq (A6)
a = mu / (mu**2 - 1)        # eq (A5)
max(min(a, 0.5), -0.5)      # eq (A3)

np.float64(-0.32967032967032983)

如果要写为函数，需要留意的是相同半径的矫正系数是零。

In [8]:
def becke_radii_adjust_recap(zs, radii):
    natm = len(zs)
    fac_radii = np.zeros((natm, natm))
    for i in range(natm):
        for j in range(natm):
            rA = radii[zs[i]]
            rB = radii[zs[j]]
            if rA == rB:
                fac_radii[i, j] = 0.0
            else:
                mu = (rA - rB) / (rA + rB)
                a = mu / (mu**2 - 1)
                fac_radii[i, j] = max(min(a, 0.5), -0.5)
    return fac_radii

In [9]:
np.allclose(becke_radii_adjust_recap(mol.atom_charges(), grids.atomic_radii), fac_radii)

True

最后要表明，这里的 atom_charges 表示的是原子序数，不是真的电荷 (虚原子、赝电荷等情况)。

### 原始格点坐标与权重的拼接

### Becke Partition 权重计算

- `atom_dist_inv`: $1 / R_{AB}$

In [10]:
atm_coords = mol.atom_coords()
atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)
for i in range(natm):
    atom_dist[i, i] = np.inf
atom_dist_inv = 1.0 / atom_dist

$$
\begin{align}
\mu_{AB} &= \frac{r_A - r_B}{R_{AB}} \tag{11} \\
\mu_{AB} \sim \nu_{AB} &= \mu_{AB} + a_{AB} ( 1 - \mu_{AB}^2 ) \tag{A2} \\
p(\mu) &= \frac{3}{2} \mu - \frac{1}{2} \mu^3 \tag{19} \\
f_3(\mu) &= p(p(p(\mu))) \tag{20} \\
s_3(\mu) &= \frac{1}{2} ( 1 - f_3(\mu) ) \tag{21} \\
P_A (\bm{r}_g) &= \prod_{A \neq B} s_3(\nu_{AB}) \tag{13}
\end{align}
$$

In [11]:
def becke_partition_weights_scale(bare_weights, coords, hardness=3):
    ngrids = bare_weights.shape[0]
    weights_scale = np.ones((natm, ngrids))
    r_table = np.zeros((natm, ngrids))
    for A in range(natm):
        r_table[A] = np.linalg.norm(coords - mol.atom_coord(A), axis=-1)
    for g in range(ngrids):
        for A in range(natm):
            for B in range(A):
                rA = r_table[A, g]
                rB = r_table[B, g]
                mu = (rA - rB) * atom_dist_inv[A, B]  # eq (11)
                a = fac_radii[A, B]
                nu = mu + a * (1 - mu * mu)           # eq (A2)
                
                f = nu
                for _ in range(hardness):
                    f = 1.5 * f - 0.5 * f * f * f     # eq (19) and (20)
                sA = 0.5 * (1 - f)                    # eq (21)
                sB = 0.5 * (1 + f)                    # anti-symmetric
                weights_scale[A, g] *= sA             # eq (13)
                weights_scale[B, g] *= sB             # eq (13)
    return weights_scale

### 整合 Becke Partition：依原子迭代

$$
w_n (\bm{r}_g) = \frac{ P_A (\bm{r}_g) }{ \sum_{B} P_B (\bm{r}_g) } \tag{22}
$$

这是标准的做法。既然最后的权重缩放是依原子进行的，那么迭代就以原子为单位进行。

In [12]:
grid_coords_list = []
grid_weights_list = []
for A in range(natm):
    symbol = mol.atom_symbol(A)
    bare_coords, bare_weights = atom_grids_tab[symbol]
    bare_coords = bare_coords + mol.atom_coord(A)
    grid_coords_list.append(bare_coords)
    weights_scale = becke_partition_weights_scale(bare_weights, bare_coords)
    grid_weights_list.append(bare_weights * weights_scale[A] / weights_scale.sum(axis=0))

In [13]:
tc = np.concatenate(grid_coords_list, axis=0)
tw = np.concatenate(grid_weights_list, axis=0)

In [14]:
assert np.allclose(grids.coords[:-2], tc)
assert np.allclose(grids.weights[:-2], tw)

### 整合 Becke Partition：依原子列表索引

但是，我们在 numint 计算过程中，经常是以一个固定的 batch size 进行批处理，而不是先依原子 (如果 macro batch 是原子，那么容易导致并行不充分)。因此，我们也要备份一个依原子列表索引的整合方法。

首先我们构造格点，以及原始的权重。但在这个过程中，我们不先 partition；而是记录原子索引列表。

In [15]:
grid_coords_list = []
bare_weights_list = []
atm_idx_list = []
for A in range(natm):
    symbol = mol.atom_symbol(A)
    bare_coords, bare_weights = atom_grids_tab[symbol]
    bare_coords = bare_coords + mol.atom_coord(A)
    grid_coords_list.append(bare_coords)
    bare_weights_list.append(bare_weights)
    atm_idx_list.append(np.full(bare_coords.shape[0], A, dtype=int))
grid_coords = np.concatenate(grid_coords_list, axis=0)
bare_weights = np.concatenate(bare_weights_list, axis=0)
atm_idx = np.concatenate(atm_idx_list, axis=0)

随后分批对权重作处理：

In [16]:
ngrids = len(grid_coords)
grid_weights = np.zeros(ngrids)
for g_start in range(0, ngrids, 512):
    g_end = min(ngrids, g_start + 512)
    slc_g = slice(g_start, g_end)
    weights_scale = becke_partition_weights_scale(bare_weights[slc_g], grid_coords[slc_g])
    atm_idx_slc = atm_idx[slc_g]

    scale_numerator = np.zeros(g_end - g_start)
    for g in range(g_start, g_end):
        A = atm_idx[g]
        scale_numerator[g - g_start] = weights_scale[A, g - g_start]
    scale_denominator = weights_scale.sum(axis=0)
    grid_weights[slc_g] = bare_weights[slc_g] * scale_numerator / scale_denominator

In [17]:
np.allclose(grids.coords[:-2], grid_coords)

True

## 用于后续程序开发的方便函数

In [18]:
def becke_partition_deriv0(quadrature_weights, grid_coords, atm_coords, atm_indices, radii_table):
    """ Becke Partition: non-derivative

    Parameters
    ----------

    quadrature_weights : ndarray
        The quadrature weights (original Lebedev weights) for each grid point. Shape (ngrids,).
    grid_coords : ndarray
        The coordinates of the grid points. Shape (ngrids, 3).
    atm_coords : ndarray
        The coordinates of the atoms. Shape (natm, 3).
    atm_indices : ndarray
        The indices of the atoms corresponding to each grid point. Shape (ngrids,).
    radii_table : ndarray
        The radii table for grid partition. Shape (natm, natm).
    """
    
    # rename some inputs
    wquad = quadrature_weights
    a = radii_table

    # shapes
    natm = atm_coords.shape[0]
    ngrids = grid_coords.shape[0]

    # atomic preparations -- before grid iteration in production impl
    atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)
    for i in range(natm):
        atom_dist[i, i] = np.inf
    # atom_dist_inv = 1.0 / atom_dist  # this will be used in production code

    # closures to be used in becke partition
    # note these closures are not that efficient
    fn_nu = lambda mu, a: mu + a * (1 - mu**2)  # eq (A2)
    fn_p = lambda nu: 1.5 * nu - 0.5 * nu**3    # eq (19)
    fn_f1 = fn_p
    fn_f2 = lambda nu: fn_p(fn_f1(nu))
    fn_f3 = lambda nu: fn_p(fn_f2(nu))          # eq (20)
    fn_s3 = lambda nu: 0.5 * (1 - fn_f3(nu))    # eq (21)
    fn_s = lambda mu, a: fn_s3(fn_nu(mu, a))    # eq (A1)

    # grid-preparation -- initialized by batch in production impl
    grid_dist = np.linalg.norm(grid_coords[None, :, :] - atm_coords[:, None, :], axis=-1)  # (natm, ngrids)

    # actual computation -- probably better use iteration instead of vectorized numpy arrays
    mu = (grid_dist[:, None, :] - grid_dist[None, :, :]) / atom_dist[:, :, None]  # eq (11), (natm, natm, ngrids)
    s = fn_s(mu, a[:, :, None])  # eq (21), (natm, natm, ngrids)
    for i in range(natm):
        s[i, i, :] = 1.0  # prod (N != M)
    P = np.prod(s, axis=1)  # eq (13), (natm, ngrids)
    Z = P.sum(axis=0)  # (ngrids,)
    Pg = P[atm_indices, np.arange(ngrids)]  # (ngrids,)
    w = wquad * Pg / Z  # eq (22), (ngrids,)
    return {
        "atom_dist": atom_dist,
        "mu": mu,
        "s": s,
        "P": P,
        "Z": Z,
        "Pg": Pg,
        "w": w
    }

In [19]:
nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]

becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
natm = atm_coords.shape[0]
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)

result_deriv0 = becke_partition_deriv0(quadrature_weights, grid_coords, atm_coords, atm_indices, radii_table)

In [20]:
weights_ref = grids.weights[nonpad_mask]
assert np.allclose(result_deriv0["w"], weights_ref)